# 01 — Préparation des données Tox21

Chargement du jeu **Tox21** (MoleculeNet, 12 tests de toxicité), standardisation des molécules, **scaffold split** et calcul des descripteurs.

In [ ]:
# Colab : décommenter pour installer RDKit et cloner le dépôt
# !pip install rdkit shap xgboost pyarrow -q
# !git clone https://github.com/AymaneAmri-MD/admet-tox-prediction.git
# %cd admet-tox-prediction

In [ ]:
import sys
sys.path.append('src')
import pandas as pd
import numpy as np
from chem_utils import standardize_smiles, scaffold_split, morgan_fp, physchem

URL = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz'
df = pd.read_csv(URL)
TASKS = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER','NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53']
print(df.shape)
df.head()

## Aperçu des tâches
Les labels sont **déséquilibrés** et contiennent des valeurs manquantes : on utilisera le ROC-AUC et on ignorera les NaN à l'entraînement.

In [ ]:
summary = pd.DataFrame({
    'n_labelisées': df[TASKS].notna().sum(),
    'taux_positifs_%': (df[TASKS].mean() * 100).round(1),
})
summary

## Standardisation et nettoyage

In [ ]:
df['smiles_std'] = df['smiles'].apply(standardize_smiles)
df = df.dropna(subset=['smiles_std']).drop_duplicates('smiles_std').reset_index(drop=True)
print('Molécules après nettoyage :', len(df))

## Scaffold split (80/10/10)
Un split aléatoire surestime les performances : on sépare par **squelette moléculaire** pour tester la généralisation à de nouvelles familles chimiques.

In [ ]:
train_idx, valid_idx, test_idx = scaffold_split(df['smiles_std'].tolist())
df['split'] = 'train'
df.loc[valid_idx, 'split'] = 'valid'
df.loc[test_idx, 'split'] = 'test'
df['split'].value_counts()

## Descripteurs : empreintes de Morgan + propriétés physico-chimiques

In [ ]:
X_fp = np.vstack(df['smiles_std'].apply(morgan_fp).values)
X_pc = pd.DataFrame(df['smiles_std'].apply(physchem).tolist())
print(X_fp.shape, X_pc.shape)
X_pc.describe().round(2)

In [ ]:
import os
os.makedirs('data', exist_ok=True)
df.to_parquet('data/tox21_clean.parquet')
np.save('data/X_morgan.npy', X_fp)
X_pc.to_parquet('data/X_physchem.parquet')
print('Données sauvegardées dans data/')